# **Submission 1 Machine Learning Pipeline - Coronavirus Tweet Prediction**

### Import Library

Melakukan import library yang dibutuhkan untuk submission.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tfx.components import CsvExampleGen, StatisticsGen, SchemaGen, ExampleValidator, Transform, Trainer, Tuner, Evaluator, Pusher

from tfx.proto import example_gen_pb2, trainer_pb2, pusher_pb2
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
import tensorflow_model_analysis as tfma
import os
import pandas as pd
import numpy as np

### Set Variabel

Mendefinisikan nama pipeline dan variabel dalam proyek TFX (TensorFlow Extended). Variabel PIPELINE_NAME dan SCHEMA_PIPELINE_NAME mochyusuf-pipeline dan pipeline untuk analisis skema data coronavirus-schema. PIPELINE_ROOT adalah direktori tempat artefak pipeline disimpan, METADATA_PATH menunjuk ke lokasi file metadata yang menyimpan informasi eksekusi pipeline, dan SERVING_MODEL_DIR adalah direktori tujuan model yang sudah dilatih dan siap untuk disajikan (deployment).

In [2]:
PIPELINE_NAME = "mochyusuf-pipeline"
SCHEMA_PIPELINE_NAME = "coronavirus-schema"

PIPELINE_ROOT = PIPELINE_NAME
METADATA_PATH = os.path.join('metadata', PIPELINE_NAME, 'metadata.db')
SERVING_MODEL_DIR = os.path.join('serving_model', PIPELINE_NAME)

Membaca dataset bernama "Corona_NLP.csv" berada dalam folder "data" menggunakan library pandas kemudian disimpan ke dalam variabel corona. Lanjut info() untuk menampilkan ringkasan informasi dari DataFrame.

In [3]:
corona = pd.read_csv("data/Corona_NLP.csv", encoding='latin1')
corona.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41157 entries, 0 to 41156
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   UserName       41157 non-null  int64 
 1   ScreenName     41157 non-null  int64 
 2   Location       32567 non-null  object
 3   TweetAt        41157 non-null  object
 4   OriginalTweet  41157 non-null  object
 5   Sentiment      41157 non-null  object
dtypes: int64(2), object(4)
memory usage: 1.9+ MB


Drop kolom yang tidak digunakan yaitu kolom Username, ScreenName, Location dan TweetAt kemudian simpan di data_path value data dengan nama Corona_NLP.csv

In [4]:
data_path = "data"

corona = corona.drop(["UserName", "ScreenName", "Location", "TweetAt"], axis = 1)

corona.to_csv(os.path.join(data_path, "Corona_NLP.csv"), index = False)


Inisalisasi DATA_ROOT dengan value data kemudian setup metadata ke PIPELINE_ROOT

In [5]:
DATA_ROOT = 'data'
interactive_context = InteractiveContext(pipeline_root=PIPELINE_ROOT)

### Data Ingestion

Mengimpor dan membagi data untuk pipeline

In [6]:
output = example_gen_pb2.Output(
    split_config = example_gen_pb2.SplitConfig(splits=[
        example_gen_pb2.SplitConfig.Split(name="train", hash_buckets=8),
        example_gen_pb2.SplitConfig.Split(name="eval", hash_buckets=2)
    ])
)
example_gen = CsvExampleGen(input_base=DATA_ROOT, output_config=output)
interactive_context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 282
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}))

### Data Validation

Menghasilkan dan menampilkan statistik deskriptif dari data

In [7]:
statistics_gen = StatisticsGen(examples=example_gen.outputs["examples"])
interactive_context.run(statistics_gen)

ExecutionResult(
    component_id: StatisticsGen
    execution_id: 283
    outputs:
        statistics: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={}))

Menampilkan statistik pada dataset.

In [8]:
interactive_context.show(statistics_gen.outputs["statistics"])

Menampilkan schema pada dataset.

In [9]:
schema_gen = SchemaGen(statistics=statistics_gen.outputs["statistics"])
interactive_context.run(schema_gen)

ExecutionResult(
    component_id: SchemaGen
    execution_id: 284
    outputs:
        schema: OutputChannel(artifact_type=Schema, producer_component_id=SchemaGen, output_key=schema, additional_properties={}, additional_custom_properties={}))

Membuat SchemaGen untuk memperoleh informasi dari feature OriginalTweet dan Sentiment

In [10]:
interactive_context.show(schema_gen.outputs["schema"])

,Type,Presence,Valency,Domain
Feature name,,,,
'OriginalTweet',BYTES,required,,-
'Sentiment',STRING,required,,'Sentiment'


,Values
Domain,
'Sentiment',"'Extremely Negative', 'Extremely Positive', 'Negative', 'Neutral', 'Positive'"


In [11]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"], schema=schema_gen.outputs["schema"]
)
interactive_context.run(example_validator)

ExecutionResult(
    component_id: ExampleValidator
    execution_id: 285
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}))

Menampilkan anomali pada dataset.

In [12]:
interactive_context.show(example_validator.outputs["anomalies"])

### Data Preprocessing

Menjalankan Transform di file coronavirus_transform.py, yang bertujuan untuk melakukan preprocessing dan transformasi data menggunakan modul yang telah dibuat. Tujuan transform untuk mengubah data menjadi format yang siap untuk digunakan dalam model pelatihan.

In [13]:
TRANSFORM_MODULE_FILE = "coronavirus_transform.py"
transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs['schema'],
    module_file=os.path.abspath(TRANSFORM_MODULE_FILE)
)
interactive_context.run(transform)

Instructions for updating:
Use ref() instead.


Instructions for updating:
Use ref() instead.


INFO:tensorflow:Assets written to: mochyusuf-pipeline\Transform\transform_graph\286\.temp_path\tftransform_tmp\3cef07ff60dd4f87bb8cdbadc7d4d632\assets


INFO:tensorflow:Assets written to: mochyusuf-pipeline\Transform\transform_graph\286\.temp_path\tftransform_tmp\3cef07ff60dd4f87bb8cdbadc7d4d632\assets


INFO:tensorflow:struct2tensor is not available.


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 286
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={})
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={})
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={})
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={})
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={})
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={})
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={})
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}))

### Model Tuning, Training, Analysis, and Evaluation

Tuning hyperparameter pada model agar mendapatkan model yang terbaik dengan menggunakan tuner di file coronavirus_tuner.py.

In [14]:
TUNER_MODULE_FILE = "coronavirus_tuner.py"
tuner = Tuner(
    module_file=os.path.abspath(TUNER_MODULE_FILE),
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(splits=['train']),
    eval_args=trainer_pb2.EvalArgs(splits=['eval']),
)
interactive_context.run(tuner)

Trial 10 Complete [00h 01m 09s]
val_accuracy: 0.687741756439209

Best val_accuracy So Far: 0.7034574747085571
Total elapsed time: 00h 10m 22s
Results summary
Results in mochyusuf-pipeline\.temp\287\coronavirus_sentiment_random_search
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 06 summary
Hyperparameters:
unit_1: 128
unit_2: 64
unit_3: 16
Score: 0.7034574747085571

Trial 02 summary
Hyperparameters:
unit_1: 64
unit_2: 32
unit_3: 16
Score: 0.6987427473068237

Trial 04 summary
Hyperparameters:
unit_1: 256
unit_2: 64
unit_3: 16
Score: 0.6978965401649475

Trial 05 summary
Hyperparameters:
unit_1: 256
unit_2: 32
unit_3: 64
Score: 0.6951160430908203

Trial 03 summary
Hyperparameters:
unit_1: 64
unit_2: 128
unit_3: 32
Score: 0.6931818127632141

Trial 07 summary
Hyperparameters:
unit_1: 128
unit_2: 32
unit_3: 64
Score: 0.6929400563240051

Trial 01 summary
Hyperparameters:
unit_1: 256
unit_2: 32
unit_3: 32
Score: 0.6917311549186707

Trial 09 summary
Hyperparamete

ExecutionResult(
    component_id: Tuner
    execution_id: 287
    outputs:
        best_hyperparameters: OutputChannel(artifact_type=HyperParameters, producer_component_id=Tuner, output_key=best_hyperparameters, additional_properties={}, additional_custom_properties={})
        tuner_results: OutputChannel(artifact_type=TunerResults, producer_component_id=Tuner, output_key=tuner_results, additional_properties={}, additional_custom_properties={}))

Melatih model dengan menggunakan hyperparameter dari hasil tuning

In [15]:
TRAINER_MODULE_FILE = "coronavirus_trainer.py"
trainer = Trainer(
    module_file=os.path.abspath(TRAINER_MODULE_FILE),
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    hyperparameters=tuner.outputs['best_hyperparameters'],
    train_args=trainer_pb2.TrainArgs(splits=['train']),
    eval_args=trainer_pb2.EvalArgs(splits=['eval']),
)
interactive_context.run(trainer)

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 OriginalTweet_xf (InputLaye  [(None, 1)]              0         
 r)                                                              
                                                                 
 tf.reshape (TFOpLambda)     (None,)                   0         
                                                                 
 text_vectorization (TextVec  (None, 100)              0         
 torization)                                                     
                                                                 
 embedding (Embedding)       (None, 100, 16)           160000    
                                                                 
 global_average_pooling1d_1   (None, 16)               0         
 (GlobalAveragePooling1D)                                        
                                                           

INFO:tensorflow:Assets written to: mochyusuf-pipeline\Trainer\model\288\Format-Serving\assets


INFO:tensorflow:Assets written to: mochyusuf-pipeline\Trainer\model\288\Format-Serving\assets


1000/1000 [==============================] - 9s 8ms/step - loss: 1.4333 - accuracy: 0.3532 - val_loss: 1.3095 - val_accuracy: 0.4379 - lr: 0.0010
Epoch 2/10
 989/1000 [============================>.] - ETA: 0s - loss: 1.1425 - accuracy: 0.5233
Epoch 2: val_accuracy improved from 0.43792 to 0.59909, saving model to mochyusuf-pipeline\Trainer\model\288\Format-Serving


INFO:tensorflow:Assets written to: mochyusuf-pipeline\Trainer\model\288\Format-Serving\assets


...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: mochyusuf-pipeline\Trainer\model\288\Format-Serving\assets


INFO:tensorflow:Assets written to: mochyusuf-pipeline\Trainer\model\288\Format-Serving\assets


ExecutionResult(
    component_id: Trainer
    execution_id: 288
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={})
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}))

Mengambil model terbaru yang sudah blessed

In [16]:
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
).with_id('Latest_blessed_model_resolver')
interactive_context.run(model_resolver)

ExecutionResult(
    component_id: Latest_blessed_model_resolver
    execution_id: 289
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Latest_blessed_model_resolver, output_key=model, additional_properties={}, additional_custom_properties={})
        model_blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Latest_blessed_model_resolver, output_key=model_blessing, additional_properties={}, additional_custom_properties={}))

Mengevaluasi performa model dengan metrik ExampleCount, AUC, FalsePositives, TruePositives, FalseNegatives, dan TrueNegatives.

In [17]:
eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(
            signature_name="serving_default",
            label_key="Sentiment_xf",
            preprocessing_function_names=["transform_features"],
        )
    ],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(class_name="ExampleCount"),
                tfma.MetricConfig(class_name="AUC"),
                tfma.MetricConfig(class_name="FalsePositives"),
                tfma.MetricConfig(class_name="TruePositives"),
                tfma.MetricConfig(class_name="FalseNegatives"),
                tfma.MetricConfig(class_name="TrueNegatives"),
                tfma.MetricConfig(
                    class_name="CategoricalAccuracy",
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={"value": 0.5}
                        )
                    ),
                ),
            ]
        )
    ],
)

evaluator = Evaluator(
    examples=example_gen.outputs["examples"],
    model=trainer.outputs["model"],
    baseline_model=model_resolver.outputs["model"],
    eval_config=eval_config,
)
interactive_context.run(evaluator)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 290
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={})
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}))

Menampilkan hasil evaluasi model dengan metrik yang telah ditentukan

In [18]:
eval_result = evaluator.outputs['evaluation'].get()[0].uri
tfma_result = tfma.load_eval_result(eval_result)
tfma.view.render_slicing_metrics(tfma_result)
tfma.addons.fairness.view.widget_view.render_fairness_indicator(tfma_result)

FairnessIndicatorViewer(slicingMetrics=[{'sliceValue': 'Overall', 'slice': 'Overall', 'metrics': {'accuracy': …

Melakukan proses pusher untuk model yang telah mendapatkan blessing dari evaluator

In [19]:
pusher = Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory="serving_model_dir/coronavirus-prediction-model"
        )
    )
)
interactive_context.run(pusher)

ExecutionResult(
    component_id: Pusher
    execution_id: 291
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}))